In [5]:
import numpy as np
import pandas as pd
from math import sqrt

rng = np.random.default_rng(123)

n_per_group = 8_000
n = 2 * n_per_group
variant = np.array(['control'] * n_per_group + ['treatment'] * n_per_group)
is_treatment = (variant == 'treatment').astype(int)

x_pre = rng.lognormal(mean=1.8, sigma=0.75, size=n)

post_activity = 0.8 * x_pre + 0.6 * is_treatment + rng.normal(0, 1.0, size=n)

noise = rng.normal(0, 3.5, size=n)
true_effect = 0.25
y = 3.0 + 0.65 * x_pre + true_effect * is_treatment + noise

exp = pd.DataFrame({
    'user_id': np.arange(1, n + 1),
    'variant': variant,
    'x_pre': x_pre,
    'post_activity': post_activity,
    'y': y,
})

exp.head()

,user_id,variant,x_pre,post_activity,y
0,1,control,2.881062,2.025478,5.694006
1,2,control,4.591285,2.380674,-0.058244
2,3,control,15.894036,12.061764,7.288862
3,4,control,6.996995,7.353723,11.059576
4,5,control,12.063365,8.409813,4.642669


In [6]:
#@title 🔒 Автопроверка CUPED — запустите один раз и не редактируйте
import numpy as _np
import pandas as _pd
from math import sqrt as _sqrt


def _effect_and_se(data, metric):
    c = data.loc[data['variant'] == 'control', metric]
    t = data.loc[data['variant'] == 'treatment', metric]
    effect = t.mean() - c.mean()
    se = _sqrt(t.var(ddof=1) / len(t) + c.var(ddof=1) / len(c))
    return effect, se


def _theta_reference(data, x_col='x_pre', y_col='y'):
    x = data[x_col]
    y = data[y_col]
    x_centered = x - x.mean()
    y_centered = y - y.mean()
    cov_y_x = (x_centered * y_centered).sum() / (len(data) - 1)
    var_x = (x_centered ** 2).sum() / (len(data) - 1)
    return cov_y_x / var_x


def check_task(task_id, **kwargs):
    if task_id == 1:
        theta = kwargs.get('theta')
        ref = _theta_reference(exp)
        assert isinstance(theta, (int, float, _np.floating)), 'theta должен быть числом.'
        assert abs(theta - ref) < 1e-6, 'theta посчитан неверно. Используйте Cov(Y, X) / Var(X).'
        print('✅ Задача 1 зачтена: theta посчитан корректно.')

    elif task_id == 2:
        y_cuped = kwargs.get('y_cuped')
        assert y_cuped is not None, 'Передайте y_cuped в check_task.'
        y_cuped = _pd.Series(y_cuped)
        assert len(y_cuped) == len(exp), 'y_cuped должен иметь ту же длину, что exp.'
        ref_theta = _theta_reference(exp)
        ref = exp['y'] - ref_theta * (exp['x_pre'] - exp['x_pre'].mean())
        assert _np.allclose(y_cuped.values, ref.values), 'Формула y_cuped неверна. Проверьте центрирование X.'
        print('✅ Задача 2 зачтена: CUPED-метрика построена корректно.')

    elif task_id == 3:
        y_cuped = kwargs.get('y_cuped')
        data = exp.copy()
        data['y_cuped'] = y_cuped
        _, se_base = _effect_and_se(data, 'y')
        _, se_cuped = _effect_and_se(data, 'y_cuped')
        assert se_cuped < se_base, 'SE после CUPED должен быть меньше baseline SE.'
        reduction = 1 - se_cuped / se_base
        assert reduction > 0.15, 'Снижение SE слишком маленькое: проверьте формулу CUPED.'
        print(f'✅ Задача 3 зачтена: SE снизился на {reduction:.1%}.')

    elif task_id == 4:
        answer = str(kwargs.get('answer', '')).lower()
        explanation = str(kwargs.get('explanation', '')).lower()
        text = answer + ' ' + explanation
        assert 'post_activity' in text, 'Нужно явно назвать post_activity.'
        assert any(word in text for word in ['после', 'post', 'treatment', 'эксперимент']), 'Объясните, что признак измерен после начала эксперимента или зависит от treatment.'
        assert any(word in text for word in ['нельзя', 'некоррект', 'смещ', 'bias', 'invalid']), 'Нужно явно написать, что такой covariate использовать нельзя.'
        print('✅ Задача 4 зачтена: риск post-treatment covariate объяснен.')

    elif task_id == 5:
        rho = kwargs.get('rho')
        predicted_reduction = kwargs.get('predicted_reduction')
        ref_rho = exp['x_pre'].corr(exp['y'])
        ref_red = ref_rho ** 2
        assert abs(rho - ref_rho) < 1e-6, 'rho должен быть corr(x_pre, y).'
        assert abs(predicted_reduction - ref_red) < 1e-6, 'Ожидаемое снижение дисперсии равно rho^2.'
        print('✅ Задача 5 зачтена: связь корреляции и снижения дисперсии посчитана.')

    elif task_id == 6:
        report = kwargs.get('report')
        assert isinstance(report, _pd.DataFrame), 'report должен быть pandas DataFrame.'
        required = {'estimate', 'standard_error', 'ci_left_95', 'ci_right_95'}
        assert required.issubset(set(report.columns)), f'report должен содержать колонки {required}.'
        assert {'baseline', 'cuped'}.issubset(set(report.index)), 'Индекс report должен содержать baseline и cuped.'
        assert report.loc['cuped', 'standard_error'] < report.loc['baseline', 'standard_error'], 'CUPED SE должен быть меньше baseline SE.'
        print('✅ Задача 6 зачтена: итоговая таблица собрана корректно.')
    else:
        raise ValueError('Неизвестный номер задачи.')

## Задача 1. Посчитать коэффициент $\theta$

Посчитайте коэффициент

$$
\theta
=
\frac{
\operatorname{Cov}(Y, X)
}{
\operatorname{Var}(X)
}
$$

где:

- $Y$ — столбец `exp['y']`;
- $X$ — столбец `exp['x_pre']`.

**Требование:** посчитать коэффициент вручную через центрирование, без использования `np.cov`.

$Cov(X, Y) = \sum_{i=1}^N \frac{(x_i - \mu_X)(y_i-\mu_Y)}{N}$

$\sigma^2(X) = \frac{\sum_{i=1}^N (x_i - \mu_X)^2}{N}$

x_centered = $x_i - \mu_X$ 

y_centered = $y_i-\mu_Y$




In [8]:
# TODO: заполните код
x_centered = exp['x_pre'] - exp['x_pre'].mean()
y_centered = exp['y'] - exp['y'].mean()

cov_y_x = (x_centered * y_centered).sum() / (len(exp) - 1)
var_x = (x_centered ** 2).sum() / (len(exp) - 1)

theta = cov_y_x / var_x

check_task(1, theta=theta)

✅ Задача 1 зачтена: theta посчитан корректно.


## Задача 2. Построить CUPED-метрику

Постройте столбец `y_cuped` по формуле

$$
Y_{\mathrm{CUPED}}
=
Y
-
\theta\left(X-\bar{X}\right)
$$

где:

- $Y$ — исходная метрика;
- $X$ — pre-period метрика;
- $\bar{X}$ — среднее значение $X$;
- $\theta$ — коэффициент CUPED-коррекции.

**Важно:** перед применением формулы переменную $X$ нужно центрировать.

In [11]:
# TODO: заполните код
y_cuped = exp['y'] - theta * (exp['x_pre'] - exp['x_pre'].mean())

check_task(2, y_cuped=y_cuped)

✅ Задача 2 зачтена: CUPED-метрика построена корректно.


## Задача 3. Сравнить standard error

Посчитайте baseline effect/SE и CUPED effect/SE. Убедитесь, что CUPED дает меньшую standard error.

baseline: $\widehat{\tau}=\bar{Y}_{\mathrm{treatment}}-\bar{Y}_{\mathrm{control}}$

$\operatorname{SE}(\widehat{\tau})=\sqrt{
\frac{s_{\mathrm{treatment}}^2}{n_{\mathrm{treatment}}}+
\frac{s_{\mathrm{control}}^2}{n_{\mathrm{control}}}}$

In [29]:
# TODO: заполните код
work = exp.copy()
work['y_cuped'] = y_cuped

control_y = work.loc[exp['variant'] == 'control', 'y']
treatment_y = work.loc[exp['variant'] == 'treatment', 'y']

baseline_effect = treatment_y.mean() - control_y.mean()
baseline_se = sqrt(treatment_y.var(ddof=1) / treatment_y.shape[0] + control_y.var(ddof=1) / control_y.shape[0])

control_cuped = work.loc[work['variant'] == 'control', 'y_cuped']
treatment_cuped = work.loc[work['variant'] == 'treatment', 'y_cuped']

cuped_effect = treatment_cuped.mean() - control_cuped.mean()
cuped_se = sqrt(treatment_cuped.var(ddof=1) / treatment_cuped.shape[0] + control_cuped.var(ddof = 1) / control_cuped.shape[0])

pd.DataFrame({
    'estimate': [baseline_effect, cuped_effect],
    'standard_error': [baseline_se, cuped_se]
}, index=['baseline', 'cuped'])

check_task(3, y_cuped=y_cuped)

✅ Задача 3 зачтена: SE снизился на 38.1%.


In [32]:
work

,user_id,variant,x_pre,post_activity,y,y_cuped
0,1,control,2.881062,2.025478,5.694006,9.022863
1,2,control,4.591285,2.380674,-0.058244,2.169777
2,3,control,15.894036,12.061764,7.288862,2.241540
3,4,control,6.996995,7.353723,11.059576,11.739092
4,5,control,12.063365,8.409813,4.642669,2.061069
...,...,...,...,...,...,...
15995,15996,treatment,15.241605,13.236139,1.764648,-2.862719
15996,15997,treatment,8.014984,6.573922,12.416675,12.440933
15997,15998,treatment,10.186379,8.368847,9.119537,7.746113
15998,15999,treatment,5.918622,6.366057,7.117669,8.491311


## Задача 4. Найти неправильный covariate

В данных есть `post_activity`. Объясните, почему его нельзя использовать как CUPED-covariate.

Ответ должен явно упоминать `post_activity`.

In [31]:
answer = """
post_activity
"""

explanation = """
Если случайно в X захватить данные, которые собраны после старта теста (post_activity), в X просочится нформация о самом эксперименте, т.е. фича начнет влиять не только на Y, но и на предэкспериментальную X. Поэтому этот столбец использовать нельзя."""

check_task(4, answer=answer, explanation=explanation)

✅ Задача 4 зачтена: риск post-treatment covariate объяснен.


## Задача 5. Связать корреляцию и снижение дисперсии

Посчитайте `rho = corr(x_pre, y)` и теоретическое ожидаемое снижение дисперсии `rho ** 2`.

$\frac{\operatorname{Var}\left(Y_{\mathrm{CUPED}}\right)}{\operatorname{Var}(Y) } \approx 1-\rho^2$

In [38]:
# TODO: заполните код
rho = work['x_pre'].corr(work['y'])
predicted_reduction = 1 - work['y_cuped'].var(ddof=1) / work['y'].var(ddof=1)

check_task(5, rho=rho, predicted_reduction=predicted_reduction)

✅ Задача 5 зачтена: связь корреляции и снижения дисперсии посчитана.


## Задача 6. Собрать итоговый отчет

Соберите `report` с индексами `baseline` и `cuped` и колонками:

- `estimate`
- `standard_error`
- `ci_left_95`
- `ci_right_95`

Используйте нормальное приближение `estimate ± 1.96 * standard_error`.

In [40]:
# TODO: заполните код
report = pd.DataFrame({
    'estimate': [baseline_effect, cuped_effect],
    'standard_error': [baseline_se, cuped_se],
    'ci_left_95': [baseline_effect - 1.96 * baseline_se, cuped_effect - 1.96 * cuped_se],
    'ci_right_95': [baseline_effect - 1.96 * baseline_se, cuped_effect + 1.96 * cuped_se],
}, index=['baseline', 'cuped'])

report

check_task(6, report=report)

✅ Задача 6 зачтена: итоговая таблица собрана корректно.
